<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-35.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 35 - Integridad, Checksums y Migración de Datos

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Objetivo

En este notebook trabajaremos con:

1. **Checksums**: calcular y verificar hashes MD5 y SHA-256
2. **Migración de datos**: simular una migración con verificación de integridad
3. **Metadatos**: registrar el linaje y la calidad de los datos
4. **Ejercicios de práctica** para la evaluación BDA-RA3

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import os
import json
import shutil
import time
from datetime import datetime

np.random.seed(42)
print("Librerías cargadas correctamente ✅")

---

## 1. Fundamentos de checksums

Un **checksum** es una huella digital de un dato. Si el dato cambia, aunque sea un solo bit, el checksum cambia completamente.

In [ ]:
texto_original = "Hola Mundo"
texto_modificado = "Hola mundo"  # Solo cambia M → m

print("=" * 60)
print("DEMOSTRACIÓN: Efecto avalancha en funciones hash")
print("=" * 60)

for texto, etiqueta in [(texto_original, "Original"), (texto_modificado, "Modificado (M→m)")]:
    datos = texto.encode('utf-8')
    md5 = hashlib.md5(datos).hexdigest()
    sha256 = hashlib.sha256(datos).hexdigest()

    print(f"\n📝 Texto {etiqueta}: '{texto}'")
    print(f"   MD5:    {md5}")
    print(f"   SHA-256: {sha256}")

print("\n⚡ Observa: un solo carácter diferente genera hashes completamente distintos")
print("   Esto es el 'efecto avalancha', fundamental para detectar cualquier alteración")

---

## 2. Checksums de archivos

En la práctica, calculamos checksums de archivos completos para verificar que no se han corrompido.

In [ ]:
os.makedirs('integridad', exist_ok=True)

def calcular_checksum_archivo(ruta, algoritmo='sha256'):
    """Calcula el checksum de un archivo leyendo por bloques (eficiente para archivos grandes)."""
    h = hashlib.new(algoritmo)
    with open(ruta, 'rb') as f:
        while True:
            bloque = f.read(8192)
            if not bloque:
                break
            h.update(bloque)
    return h.hexdigest()

def calcular_checksum_dataframe(df, algoritmo='sha256'):
    """Calcula un checksum del contenido de un DataFrame."""
    datos = df.to_csv(index=False).encode('utf-8')
    return hashlib.new(algoritmo, datos).hexdigest()

# Crear dataset de prueba
N = 10000
df_origen = pd.DataFrame({
    'id': range(1, N + 1),
    'valor': np.random.normal(100, 25, N).round(2),
    'categoria': np.random.choice(['A', 'B', 'C', 'D'], N),
    'fecha': pd.date_range('2025-01-01', periods=N, freq='h').astype(str)
})

# Guardar en diferentes formatos
df_origen.to_csv('integridad/datos_origen.csv', index=False)
df_origen.to_parquet('integridad/datos_origen.parquet', index=False)
df_origen.to_json('integridad/datos_origen.json', orient='records', lines=True)

print("=" * 60)
print("CHECKSUMS DE ARCHIVOS")
print("=" * 60)

for archivo in ['datos_origen.csv', 'datos_origen.parquet', 'datos_origen.json']:
    ruta = f'integridad/{archivo}'
    tamano = os.path.getsize(ruta)
    md5 = calcular_checksum_archivo(ruta, 'md5')
    sha256 = calcular_checksum_archivo(ruta, 'sha256')
    print(f"\n📄 {archivo} ({tamano/1024:.1f} KB)")
    print(f"   MD5:    {md5}")
    print(f"   SHA-256: {sha256[:32]}...")

checksum_df = calcular_checksum_dataframe(df_origen)
print(f"\n📊 Checksum del DataFrame en memoria:")
print(f"   SHA-256: {checksum_df[:32]}...")

---

## 3. Simulación de migración de datos

Simulamos una migración completa: origen → destino, con verificación de integridad en cada paso.

In [ ]:
def migrar_con_verificacion(origen_dir, destino_dir):
    """Simula una migración de datos con verificación de integridad."""
    os.makedirs(destino_dir, exist_ok=True)
    informe = {
        'inicio': datetime.now().isoformat(),
        'origen': origen_dir,
        'destino': destino_dir,
        'archivos': [],
        'errores': []
    }

    archivos = [f for f in os.listdir(origen_dir) if os.path.isfile(os.path.join(origen_dir, f))]

    for archivo in archivos:
        ruta_origen = os.path.join(origen_dir, archivo)
        ruta_destino = os.path.join(destino_dir, archivo)

        checksum_antes = calcular_checksum_archivo(ruta_origen)
        shutil.copy2(ruta_origen, ruta_destino)
        checksum_despues = calcular_checksum_archivo(ruta_destino)

        integridad_ok = checksum_antes == checksum_despues

        registro = {
            'archivo': archivo,
            'tamano_bytes': os.path.getsize(ruta_origen),
            'checksum_origen': checksum_antes,
            'checksum_destino': checksum_despues,
            'integridad': 'OK' if integridad_ok else 'FALLO'
        }
        informe['archivos'].append(registro)

        if not integridad_ok:
            informe['errores'].append(archivo)

    informe['fin'] = datetime.now().isoformat()
    informe['total_archivos'] = len(archivos)
    informe['archivos_ok'] = len(archivos) - len(informe['errores'])
    informe['resultado'] = 'ÉXITO' if len(informe['errores']) == 0 else 'CON ERRORES'

    return informe

# Ejecutar migración
informe = migrar_con_verificacion('integridad', 'integridad_destino')

print("=" * 60)
print("INFORME DE MIGRACIÓN")
print("=" * 60)
print(f"\nResultado general: {informe['resultado']}")
print(f"Archivos migrados: {informe['archivos_ok']}/{informe['total_archivos']}")
print(f"Errores: {len(informe['errores'])}")

print("\n--- Detalle por archivo ---")
for arch in informe['archivos']:
    emoji = '✅' if arch['integridad'] == 'OK' else '❌'
    print(f"  {emoji} {arch['archivo']:30s} ({arch['tamano_bytes']/1024:.1f} KB) - {arch['integridad']}")

---

## 4. Detección de corrupción

Simulamos qué ocurre cuando un archivo se corrompe durante la transferencia.

In [ ]:
# Corromper intencionalmente un archivo en el destino
archivo_corrupto = 'integridad_destino/datos_origen.csv'
checksum_antes_corrupcion = calcular_checksum_archivo(archivo_corrupto)

with open(archivo_corrupto, 'r') as f:
    contenido = f.read()

# Cambiar un solo carácter en medio del archivo
posicion = len(contenido) // 2
contenido_corrupto = contenido[:posicion] + 'X' + contenido[posicion+1:]

with open(archivo_corrupto, 'w') as f:
    f.write(contenido_corrupto)

checksum_despues_corrupcion = calcular_checksum_archivo(archivo_corrupto)

print("=" * 60)
print("SIMULACIÓN DE CORRUPCIÓN DE DATOS")
print("=" * 60)
print(f"\nArchivo: {archivo_corrupto}")
print(f"Checksum antes:  {checksum_antes_corrupcion[:32]}...")
print(f"Checksum después: {checksum_despues_corrupcion[:32]}...")
print(f"\n{'✅ Integridad OK' if checksum_antes_corrupcion == checksum_despues_corrupcion else '❌ ¡CORRUPCIÓN DETECTADA!'}")

# Verificar todos los archivos del destino contra el origen
print("\n--- Verificación completa origen vs destino ---")
for archivo in os.listdir('integridad'):
    if not os.path.isfile(f'integridad/{archivo}'):
        continue
    ruta_origen = f'integridad/{archivo}'
    ruta_destino = f'integridad_destino/{archivo}'
    if os.path.exists(ruta_destino):
        cs_orig = calcular_checksum_archivo(ruta_origen)
        cs_dest = calcular_checksum_archivo(ruta_destino)
        ok = cs_orig == cs_dest
        emoji = '✅' if ok else '❌'
        print(f"  {emoji} {archivo}: {'INTACTO' if ok else 'CORRUPTO'}")

---

## 5. Registro de metadatos

Creamos un sistema básico de registro de metadatos para documentar las operaciones sobre los datos.

In [ ]:
def generar_metadatos(df, nombre_dataset, origen, operaciones=None):
    """Genera un registro completo de metadatos para un dataset."""
    metadatos = {
        'nombre': nombre_dataset,
        'timestamp': datetime.now().isoformat(),
        'origen': origen,
        'dimensiones': {
            'filas': len(df),
            'columnas': len(df.columns)
        },
        'schema': {},
        'calidad': {
            'nulos_por_columna': df.isnull().sum().to_dict(),
            'pct_completitud': round((1 - df.isnull().sum().sum() / df.size) * 100, 2),
            'duplicados': int(df.duplicated().sum())
        },
        'checksum': calcular_checksum_dataframe(df),
        'estadisticas': {},
        'operaciones': operaciones or []
    }

    for col in df.columns:
        metadatos['schema'][col] = {
            'tipo': str(df[col].dtype),
            'valores_unicos': int(df[col].nunique()),
            'nulos': int(df[col].isnull().sum())
        }

    for col in df.select_dtypes(include=[np.number]).columns:
        metadatos['estadisticas'][col] = {
            'media': round(float(df[col].mean()), 2),
            'mediana': round(float(df[col].median()), 2),
            'min': round(float(df[col].min()), 2),
            'max': round(float(df[col].max()), 2),
            'std': round(float(df[col].std()), 2)
        }

    return metadatos

metadatos = generar_metadatos(
    df_origen,
    'datos_prueba_v1',
    'generación sintética (numpy)',
    operaciones=['Generación de datos', 'Exportación a CSV/Parquet/JSON']
)

# Guardar metadatos
with open('integridad/metadatos.json', 'w', encoding='utf-8') as f:
    json.dump(metadatos, f, indent=2, ensure_ascii=False)

print("=" * 60)
print("REGISTRO DE METADATOS")
print("=" * 60)
print(json.dumps(metadatos, indent=2, ensure_ascii=False))

---

## 6. Ejercicios de práctica para la evaluación

Estos ejercicios son similares a los que aparecerán en la evaluación BDA-RA3.

In [ ]:
# EJERCICIO 1: Verificar integridad de una transferencia
# Se te proporcionan dos DataFrames: uno "original" y uno "recibido".
# Determina si la transferencia fue exitosa.

df_original = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'nombre': ['Ana', 'Carlos', 'María', 'Pedro', 'Laura'],
    'salario': [30000, 45000, 38000, 52000, 41000]
})

df_recibido = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'nombre': ['Ana', 'Carlos', 'María', 'Pedro', 'Laura'],
    'salario': [30000, 45000, 38000, 52000, 41000]
})

cs_original = calcular_checksum_dataframe(df_original)
cs_recibido = calcular_checksum_dataframe(df_recibido)

print("EJERCICIO 1: Verificación de transferencia")
print(f"Checksum original: {cs_original[:32]}...")
print(f"Checksum recibido: {cs_recibido[:32]}...")
print(f"Resultado: {'✅ Transferencia exitosa' if cs_original == cs_recibido else '❌ Datos alterados'}")

# Ahora con un dato alterado
df_recibido_malo = df_recibido.copy()
df_recibido_malo.loc[2, 'salario'] = 38001  # Un euro de diferencia

cs_recibido_malo = calcular_checksum_dataframe(df_recibido_malo)
print(f"\nCon 1€ de diferencia en un salario:")
print(f"Checksum alterado: {cs_recibido_malo[:32]}...")
print(f"Resultado: {'✅ Transferencia exitosa' if cs_original == cs_recibido_malo else '❌ Datos alterados'}")

In [ ]:
# EJERCICIO 2: Pipeline completo de migración
# Implementa un pipeline que:
# 1. Genere un dataset
# 2. Calcule checksums
# 3. "Migre" los datos (con posible corrupción)
# 4. Verifique la integridad
# 5. Genere un informe

print("\nEJERCICIO 2: Pipeline de migración completo")
print("=" * 50)

# Paso 1: Generar dataset
df_migracion = pd.DataFrame({
    'id': range(1, 1001),
    'temperatura': np.round(np.random.normal(22, 5, 1000), 1),
    'humedad': np.round(np.random.uniform(30, 90, 1000), 1),
    'presion': np.round(np.random.normal(1013, 10, 1000), 1)
})

# Paso 2: Guardar y calcular checksums
os.makedirs('migracion/origen', exist_ok=True)
os.makedirs('migracion/destino', exist_ok=True)
df_migracion.to_parquet('migracion/origen/sensores.parquet', index=False)
cs_origen = calcular_checksum_archivo('migracion/origen/sensores.parquet')

# Paso 3: Migrar
shutil.copy2('migracion/origen/sensores.parquet', 'migracion/destino/sensores.parquet')
cs_destino = calcular_checksum_archivo('migracion/destino/sensores.parquet')

# Paso 4: Verificar
integridad = cs_origen == cs_destino

# Paso 5: Verificar conteos y sumas
df_verificacion = pd.read_parquet('migracion/destino/sensores.parquet')
conteo_ok = len(df_migracion) == len(df_verificacion)
suma_ok = abs(df_migracion['temperatura'].sum() - df_verificacion['temperatura'].sum()) < 0.01

print(f"\n1️⃣ Dataset generado: {len(df_migracion):,} registros")
print(f"2️⃣ Checksum origen:  {cs_origen[:32]}...")
print(f"3️⃣ Migración completada")
print(f"4️⃣ Checksum destino: {cs_destino[:32]}...")
print(f"\n--- VERIFICACIONES ---")
print(f"  {'✅' if integridad else '❌'} Integridad de archivo (SHA-256)")
print(f"  {'✅' if conteo_ok else '❌'} Conteo de registros ({len(df_verificacion):,})")
print(f"  {'✅' if suma_ok else '❌'} Checksum numérico (sum temperatura)")
print(f"\n📊 Resultado: {'MIGRACIÓN EXITOSA ✅' if all([integridad, conteo_ok, suma_ok]) else 'MIGRACIÓN FALLIDA ❌'}")

---

## 7. Resumen y preparación para la evaluación

### Conceptos clave para la evaluación BDA-RA3:

| Tema | Conceptos |
|------|-----------|
| **Calidad de datos** | 6 dimensiones, quality score, profiling, técnicas de limpieza |
| **Checksums** | MD5, SHA-256, efecto avalancha, verificación de integridad |
| **Migración** | Estrategias (big bang, incremental, paralela), verificación post-migración |
| **Metadatos** | Tipos (técnicos, operacionales, de negocio), catálogos, linaje |

### Funciones Python esenciales:
- `hashlib.sha256(data).hexdigest()` → checksum
- `df.isnull().sum()` → detectar nulos
- `df.duplicated()` → detectar duplicados
- `df.describe()` → profiling rápido
- `json.dump()` → guardar metadatos

---

## Ejercicios adicionales

1. Implementa un verificador que compare dos directorios completos archivo por archivo.
2. Crea un sistema de "versiones" de datos usando checksums (como un mini git para datos).
3. Investiga: ¿cómo funciona el checksum CRC32 que usa HDFS internamente?